In [1]:
import numpy as np
from compressor_gpu import Compressor

In [2]:
data = np.random.rand(1000, 3)
cp = Compressor(data)

In [ ]:
weights = cp.compress_weights(2, return_at=[300,400])

Compressing:   0%|          | 0/1000 [00:00<?, ?pt/s]


In [3]:
import numpy as np
import torch

def null_space_gpu(A: np.ndarray, rcond: float = 1e-12) -> np.ndarray:
    """
    Compute an approximate basis for the null space of A using GPU-accelerated SVD.
    
    Parameters
    ----------
    A : np.ndarray, shape (m, n)
        Input matrix.
    rcond : float
        Relative tolerance for small singular values. Singular values S[i] <= rcond * max(S)
        are treated as zero.
    
    Returns
    -------
    Z : np.ndarray, shape (n, k)
        Orthonormal basis for the null space of A (columns).
    """
    # pick the best device
    if torch.cuda.is_available():
        device = 'cuda'
    else:
        device = 'cpu'

    # move data to torch
    X = torch.from_numpy(A).to(device=device, dtype=torch.float64)

    # full_matrices=False for economy: shapes U:(m, k), Vh:(k, n), where k=min(m,n)
    U, S, Vh = torch.linalg.svd(X, full_matrices=True)

    # determine tolerance
    tol = rcond * S.max()

    # number of non-zero singular values = rank
    rank = int((S > tol).sum().item())
    nullity = Vh.shape[0] - rank
    if nullity <= 0:
        # no null space
        return np.zeros((A.shape[1], 0))

    # null space basis = last `nullity` rows of Vh, transposed to shape (n, nullity)
    Z = Vh[rank:, :].T

    return Z.cpu().numpy()

# a 3×4 matrix of rank 2
A = np.random.rand(1000, 1011)
# A = np.array([[1,2,3,4],
#               [2,4,6,8],
#               [1,0,1,0]], dtype=float)
Z = null_space_gpu(A, rcond=1e-12)
print(Z.shape)   # (4, 2)
# Verify A @ Z ≈ 0
print(np.linalg.norm(A @ Z))

(1011, 11)
1.0461164262522772e-13


In [8]:
A.shape

(3, 4)